# EEG Motor Imagery Classifier

This project classifies motor imagery states from EEG brain signals 
using the PhysioNet EEG Motor Movement/Imagery dataset.

We extract frequency band features from raw EEG recordings and train 
a machine learning model to distinguish imagined left hand vs right 
hand movement.

**Skills used:** Signal processing, feature extraction, classification, 
data visualization  
**Libraries:** MNE-Python, NumPy, Pandas, Scikit-learn, Matplotlib

In [1]:
# Import all libraries needed for this project
import numpy as np          # numerical computing
import pandas as pd         # data manipulation
import matplotlib.pyplot as plt  # plotting
import seaborn as sns       # statistical visualization
import mne                  # EEG signal processing

# Suppress MNE's verbose output so only warnings print
mne.set_log_level('WARNING')

# Confirm everything loaded correctly
print("MNE version:", mne.__version__)
print("All libraries imported successfully")

MNE version: 1.12.1
All libraries imported successfully


## Step 1: Load the Data

The PhysioNet EEG Motor Imagery dataset contains EEG recordings from 
109 subjects. Each subject performed and imagined opening and closing 
their left and right fists.

We load runs 6 and 10 which correspond to imagined left vs right hand 
movement — this is the core task of a brain computer interface.

In [2]:
from mne.datasets import eegbci
from mne.io import concatenate_raws, read_raw_edf

# Set the data path explicitly so MNE never prompts for input
# This points to where MNE already downloaded the data
mne.set_config('MNE_DATA', '/Users/chiebukaonwuka/mne_data')

# Define which runs to load
# Run 6 = imagined left vs right fist (first session)
# Run 10 = imagined left vs right fist (second session)
runs = [6, 10]

# Download data for subject 1 if not already downloaded
raw_fnames = eegbci.load_data(subjects=1, runs=runs, verbose=False)

# Load each file and concatenate into one continuous recording
# preload=True loads data into memory for faster processing
raw = concatenate_raws([read_raw_edf(f, preload=True, verbose=False) 
                        for f in raw_fnames])

# Print a summary of the data structure
# This shows channels, sampling frequency, duration, and more
print(raw.info)

<Info | 8 non-empty values
 bads: []
 ch_names: Fc5., Fc3., Fc1., Fcz., Fc2., Fc4., Fc6., C5.., C3.., C1.., ...
 chs: 64 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 80.0 Hz
 meas_date: 2009-08-12 16:15:00 UTC
 nchan: 64
 projs: []
 sfreq: 160.0 Hz
 subject_info: <subject_info | his_id: X, sex: 0, last_name: X>
>


## Step 2: Explore the Raw Data

Before processing anything we need to understand what the raw EEG 
signal looks like. We'll check the duration, channel names, and 
plot the raw signal to see the brain activity directly.

In [3]:
# Print total duration of the recording in seconds
duration = raw.times[-1]
print(f"Recording duration: {duration:.1f} seconds")

# Print total number of channels
print(f"Number of channels: {raw.info['nchan']}")

# Print sampling frequency
print(f"Sampling frequency: {raw.info['sfreq']} Hz")

# Print total number of time points
print(f"Total time points: {len(raw.times)}")

# Print first 10 channel names
print(f"\nFirst 10 channels: {raw.ch_names[:10]}")

Recording duration: 250.0 seconds
Number of channels: 64
Sampling frequency: 160.0 Hz
Total time points: 40000

First 10 channels: ['Fc5.', 'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.', 'Fc4.', 'Fc6.', 'C5..', 'C3..', 'C1..']
